# Noise-robust, exactly lossless RDHEI by multi-key selection: Colab companion

Code for the paper *Noise-Robust and Exactly Lossless Reversible Data Hiding in Encrypted Images via Scalable Multi-Key Selection* (S. Singh, U. Singh).

This notebook

1. installs the code (section 1),
2. embeds a message in an image by key selection and recovers image and message bit-exactly (section 2, about 5 min),
3. reruns the experiments of the paper on test set A (section 3; a quick check takes seconds, the full run several hours on a Colab CPU),
4. regenerates every table and figure of the paper from the result files and compares a new run with them (section 4).

Experiment 8 (BOSSbase, BOWS-2, UCID) has its own notebook: `notebooks/Exp8_public_datasets_Colab.ipynb`. A GPU is not needed.

## 1. Set-up

In [ ]:
REPO_URL = "https://github.com/YOUR-GITHUB-USERNAME/rdhei-multikey-selection.git"   # <-- the URL of this repository
import os, subprocess, sys
# The learned receivers (KNN, DT, RF, HGB) depend on the numpy and scikit-learn versions. The versions of the paper are installed
# here; if Colab had different ones, the runtime restarts once (Colab then reports a crash): simply run this cell again.
import numpy, sklearn
if (numpy.__version__, sklearn.__version__) != ("2.1.3", "1.6.1"):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "numpy==2.1.3", "scikit-learn==1.6.1", "reedsolo", "cryptography"], check=True)
    os.kill(os.getpid(), 9)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "reedsolo", "cryptography"], check=True)
if not os.path.isdir("/content/rdhei-multikey-selection"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/rdhei-multikey-selection"], check=True)
REPO = "/content/rdhei-multikey-selection"
os.makedirs(f"{REPO}/work", exist_ok=True); os.chdir(f"{REPO}/work")
os.environ["RDHEI_ROOT"] = REPO; os.environ["MPLBACKEND"] = "Agg"
print("numpy", numpy.__version__, "| scikit-learn", sklearn.__version__, "| working directory:", os.getcwd())


## 2. Demonstration: embed, decode, recover exactly

Loads the scheme (`rdhei/core.py`): it downloads the test images, checks them against the SHA-256 checksums used in the paper, trains the learned receivers on the ten training images (about 4–5 min) and runs its self-tests.

In [ ]:
%matplotlib inline
exec(open(f"{REPO}/rdhei/core.py").read())

The sender encrypts every $B\times B$ block with the AES-CTR stream of one of $K$ keys, chosen by the Reed–Solomon-coded message bits, and runs the closed-loop test: it picks the smallest parity overhead of the ladder for which the receiver recovers image and message exactly. The receiver here is the training-free Rule-it.

In [ ]:
import matplotlib.pyplot as plt
def demo(img, K=16, B=4, seed=0, ladder=(0.02, 0.05, 0.10, 0.20, 0.30, 0.40)):
    rng = np.random.default_rng(seed); keys = make_keys(K, seed); m = int(math.log2(K)); nb = (512 // B) ** 2; nbits = nb * m
    for rho in ladder:                                                        # closed-loop test at the sender
        lens, nsym, ks = ecc_plan(nbits, rho); perm = np.random.default_rng(1234).permutation(nbits)
        payload = rng.integers(0, 2, sum(ks) * 8, dtype=np.uint8)            # the message
        enc, sym = embed_encrypt(img, ecc_encode(payload, lens, nsym, ks, perm), keys, B, "aesctr")
        cand = all_candidates(enc, keys, B, "aesctr"); pred, marg = decode_rule_it(cand, B)          # receiver
        bits = ((pred[:, None] >> np.arange(m)[::-1]) & 1).astype(np.uint8).ravel()
        ok, cor, msg = ecc_decode(bits, np.repeat(marg, m), lens, nsym, ks, perm)
        rec = unblockify(cand[np.arange(nb), cor.reshape(nb, m) @ (1 << np.arange(m)[::-1])], B)
        exact = bool((rec == img).all()) and bool((msg == payload[:len(msg)]).all())
        print(f"rho={rho:.2f}: block errors before RS {100*(pred != sym).mean():.3f} %, exact recovery: {exact}")
        if exact: break
    print(f"net rate {sum(ks)*8/512**2:.3f} bpp (raw {nbits/512**2:.3f} bpp); image identical: {(rec == img).all()}; message identical: {(msg == payload[:len(msg)]).all()}")
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    for a, (t, z) in zip(ax, [("cover", img), (f"marked ciphertext (K={K}, {B}x{B})", enc), ("recovered", rec)]):
        a.imshow(z, cmap="gray", vmin=0, vmax=255); a.set_title(t); a.axis("off")
    plt.show()
demo(test_A["jetplane"], K=16, B=4)

## 3. Rerun the experiments (test set A)

`experiments/run_all.sh` runs every experiment of the paper on test set A and writes one CSV per table to `work/results`.

* `quick`: the compress-then-encrypt baseline and the chi-square test over 900 key sets (seconds).
* full: all experiments (several hours on a two-core CPU; `exp_p2.py`, Experiments 2–5, is the longest step). Colab may disconnect a long session; the scripts can also be run one at a time, as in the second cell.

In [ ]:
!bash {REPO}/experiments/run_all.sh quick

In [ ]:
# individual experiments (uncomment what you need); each writes its CSVs to work/results
# !python {REPO}/experiments/exp_final.py         # Experiments 1, 6, 9; Section 7 statistics and attacks; timing
# !python {REPO}/experiments/exp_p2.py            # Experiments 2-5
# !python {REPO}/experiments/exp_hgbabl.py        # Experiment 2b (receiver ablation)
# !python {REPO}/experiments/exp_noise_small.py   # Experiment 5b
# !python {REPO}/experiments/exp7_generalisation.py   # Experiment 7 (test set B)
# !bash {REPO}/experiments/run_all.sh             # everything

## 4. Tables and figures of the paper, and comparison with a new run

The scripts in `scripts/` turn the result files in `data/` (the files behind the paper) into the LaTeX tables and PDF figures of the paper. `compare_with_paper.py` compares the CSVs of a new run with them (timings are skipped, being machine dependent).

In [ ]:
for s in ["make_results", "make_sota_datasets", "make_datasets", "make_security", "make_integrity", "make_gradient"]:
    subprocess.run(["python", f"{REPO}/scripts/{s}.py"], check=True, capture_output=True)
print(sorted(os.listdir(f"{REPO}/tables"))); print(sorted(os.listdir(f"{REPO}/figures")))
!python {REPO}/scripts/compare_with_paper.py {REPO}/work/results